# Lecture 2 — Temperature: from weather to climate

## The Mediterranean: analysing real ERA5 monthly temperature data

In Lecture 1 we introduced the distinction between **weather** and **climate** and learned some basic statistical concepts.

Here we apply those ideas to a real climate dataset: **monthly ERA5 2-m air temperature over the Mediterranean region**.

The ERA5 data are gridded in latitude and longitude. We will first look at the spatial temperature field and then calculate a **latitude-weighted Mediterranean mean** to obtain a single regional climate time series.

We will investigate:

- spatial temperature patterns
- the seasonal cycle
- climatology
- temperature anomalies
- interannual variability
- the distribution of annual temperatures
- warmest and coldest years
- long-term trends
- low-frequency variability

> **Important:** this is monthly data. We are not calculating hourly or daily extremes.

## 1. Load the data

The Binder repository should contain:

```text
climate-course/
├── environment.yml
├── notebooks/
│   └── 02_temperature.ipynb
└── data/
    └── era5_monthly_t2m_med.nc
```

The dataset contains monthly 2-m air temperature over the Mediterranean region.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.stats import linregress

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 12

data_file = Path("../data/era5_monthly_t2m_med.nc")

if not data_file.exists():
    raise FileNotFoundError(
        "Course dataset not found. Make sure "
        "era5_monthly_t2m_med.nc is in ../data/."
    )

ds = xr.open_dataset(data_file)

ds

## 2. Inspect the dataset

Before analysing climate data, always check:

- dimensions
- coordinates
- variables
- units
- time coverage

This prevents us from making assumptions about the dataset structure.

In [ ]:
print("Dimensions:")
print(ds.dims)

print("\nCoordinates:")
print(list(ds.coords))

print("\nVariables:")
print(list(ds.data_vars))

print("\nTemperature variable:")
print(ds["t2m"])

print("\nUnits:", ds["t2m"].attrs.get("units"))

print("\nTime coverage:")
print(ds.valid_time.values[0], "to", ds.valid_time.values[-1])

## 3. Temperature in degrees Celsius

ERA5 temperature is stored in Kelvin.

We convert it to degrees Celsius:

\[
T_{°C} = T_K - 273.15
\]

We create a new variable so that the original data remain unchanged.

In [ ]:
temperature = ds["t2m"] - 273.15
temperature.name = "temperature"
temperature.attrs["units"] = "°C"

temperature

## 4. Spatial temperature pattern

Before reducing the data to a regional mean, let's look at the spatial structure.

We plot the temperature field for January 2025.

In [ ]:
temperature_jan2025 = temperature.sel(
    valid_time="2025-01"
)

temperature_jan2025.plot(
    cmap="coolwarm",
    cbar_kwargs={"label": "Temperature (°C)"}
)

plt.title("Mediterranean temperature — January 2025")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

### Why isn't temperature spatially uniform?

Consider:

- latitude
- elevation
- land versus ocean
- proximity to the Mediterranean Sea
- atmospheric circulation
- oceanic and atmospheric heat transport

## 5. Calculate a Mediterranean regional mean

A regional climate statistic can be obtained by averaging the grid cells within the region.

However, grid cells at different latitudes do not have the same area.

For a latitude-longitude grid, a common area-weighting approximation is:

\[
w = \cos(\phi)
\]

where \(\phi\) is latitude.

We therefore use latitude-dependent weights when calculating the Mediterranean mean.

This gives us a single temperature value for each month.

In [ ]:
# Latitude-dependent area weights
weights = np.cos(np.deg2rad(temperature.latitude))

med_temperature = temperature.weighted(weights).mean(
    dim=("latitude", "longitude")
)

med_temperature.name = "Mediterranean_temperature"

med_temperature

## 6. The Mediterranean monthly temperature record

We have now transformed the gridded temperature field into a regional climate time series.

In [ ]:
med_temperature.plot()

plt.title("Mediterranean monthly mean temperature")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.grid(alpha=0.25)
plt.show()

## 7. The seasonal cycle

Temperature has a strong and relatively predictable annual cycle.

We estimate the typical temperature for each calendar month by averaging all Januaries together, all Februaries together, and so on.

In [ ]:
monthly_climatology = med_temperature.groupby(
    "valid_time.month"
).mean()

months = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

plt.plot(months, monthly_climatology.values, marker="o")

plt.title("Mediterranean mean seasonal cycle")
plt.xlabel("Month")
plt.ylabel("Temperature (°C)")
plt.grid(alpha=0.25)
plt.show()

### Question

Why is the seasonal cycle so large compared with the temperature difference between many neighbouring years?

What physical processes produce the annual cycle of temperature?

## 8. Climate climatology: 1991–2020

A **climatology** is a statistical description of typical climate conditions over a specified reference period.

We use **1991–2020** and calculate a mean temperature for each calendar month.

In [ ]:
reference = med_temperature.sel(
    valid_time=slice("1991-01-01", "2020-12-31")
)

climatology_1991_2020 = reference.groupby(
    "valid_time.month"
).mean()

plt.plot(
    months,
    climatology_1991_2020.values,
    marker="o"
)

plt.title("Mediterranean temperature climatology, 1991–2020")
plt.xlabel("Month")
plt.ylabel("Temperature (°C)")
plt.grid(alpha=0.25)
plt.show()

## 9. Temperature anomalies

Absolute temperature is strongly affected by the season.

To identify unusually warm or cold conditions, we calculate an **anomaly** relative to the 1991–2020 climatology:

\[
T'(t) = T(t) - \overline{T}_{1991-2020}
\]

For January, for example, we compare January temperatures with the typical January temperature — not with the annual mean.

In [ ]:
climatology_for_each_month = climatology_1991_2020.sel(
    month=med_temperature.valid_time.dt.month
)

anomalies = med_temperature - climatology_for_each_month
anomalies.name = "temperature_anomaly"

anomalies.plot()

plt.axhline(0, linewidth=1)
plt.title("Mediterranean monthly temperature anomalies")
plt.xlabel("Year")
plt.ylabel("Temperature anomaly (°C)")
plt.grid(alpha=0.25)
plt.show()

### Interpretation

An anomaly of +2°C does not mean that the Mediterranean was 2°C warmer than another region.

It means that the temperature was 2°C above the temperature that is typical for that calendar month during the chosen reference period.

## 10. Annual mean temperature

The seasonal cycle can make long-term changes harder to see.

We therefore calculate one mean temperature for each year.

In [ ]:
annual_temperature = med_temperature.groupby(
    "valid_time.year"
).mean()

annual_temperature

In [ ]:
plt.plot(
    annual_temperature.year,
    annual_temperature,
    marker="o",
    markersize=3,
    linewidth=1
)

plt.title("Mediterranean annual mean temperature")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.grid(alpha=0.25)
plt.show()

## 11. Interannual variability

The annual temperature does not increase smoothly.

Even in a warming climate, individual years can be cooler or warmer than the long-term trend.

Let's look at the distribution of annual mean temperatures.

In [ ]:
plt.hist(
    annual_temperature.values,
    bins=15,
    edgecolor="black"
)

plt.xlabel("Annual mean temperature (°C)")
plt.ylabel("Number of years")
plt.title("Distribution of Mediterranean annual mean temperature")
plt.show()

## 12. Warmest and coldest years

Which years were the warmest and coldest in the record?

This connects a statistical distribution to specific years.

In [ ]:
annual_df = annual_temperature.to_dataframe(
    name="temperature"
)

warmest = annual_df.sort_values(
    "temperature", ascending=False
).head(5)

coldest = annual_df.sort_values(
    "temperature"
).head(5)

print("Five warmest years:")
display(warmest)

print("\nFive coldest years:")
display(coldest)

## 13. Long-term temperature trend

A simple first estimate of climate change is a linear trend:

\[
T(t) = a + bt
\]

where the slope \(b\) describes the rate of temperature change.

We report the trend in **°C per decade**.

In [ ]:
years = annual_temperature.year.values
values = annual_temperature.values

slope, intercept, r_value, p_value, std_err = linregress(
    years, values
)

trend_per_decade = slope * 10

print(f"Linear trend: {trend_per_decade:.2f} °C per decade")
print(f"R² = {r_value**2:.2f}")
print(f"p-value = {p_value:.3g}")

In [ ]:
trend_line = intercept + slope * years

plt.plot(
    years,
    values,
    label="Annual mean"
)

plt.plot(
    years,
    trend_line,
    linewidth=2,
    label="Linear trend"
)

plt.title("Mediterranean long-term temperature trend")
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

### Important

The linear trend is a **statistical description of long-term change**.

It is not a prediction that every future year will be warmer than the previous year.

The deviations around the trend represent shorter-term climate variability.

## 14. Long-term signal versus variability

An 11-year running mean suppresses much of the year-to-year variability and makes lower-frequency changes easier to see.

In [ ]:
running_mean = annual_temperature.rolling(
    year=11,
    center=True
).mean()

plt.plot(
    annual_temperature.year,
    annual_temperature,
    alpha=0.4,
    label="Annual mean"
)

plt.plot(
    running_mean.year,
    running_mean,
    linewidth=2,
    label="11-year running mean"
)

plt.title(
    "Mediterranean temperature: "
    "annual variability and low-frequency signal"
)
plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

# Exercises

### Exercise 1 — Change the reference period

Calculate anomalies relative to **1961–1990** instead of 1991–2020.

- How do the anomaly values change?
- Does the underlying trend change?

---

### Exercise 2 — Compare periods

Calculate the mean annual temperature for:

- 1951–1980
- 1981–2010
- 1991–2020

How much warmer is each period than the previous one?

---

### Exercise 3 — Trend sensitivity

Calculate the temperature trend separately for:

- 1950–2025
- 1980–2025
- 2000–2025

Are the estimated trends identical?

Why might estimates over shorter periods be more sensitive to variability?

---

### Exercise 4 — Climate versus weather

An individual year can be colder than the previous year even when the long-term trend is positive.

Explain why this is not contradictory.

Use the concepts:

- climate trend
- interannual variability
- seasonal cycle
- anomalies

# Key takeaways

We have followed a complete climate-analysis workflow:

**gridded temperature → spatially weighted regional mean → seasonal cycle → climatology → anomalies → annual means → variability → distribution → trend**

The central idea is:

> **Climate is not simply the average temperature. It includes the mean state, variability, distributions, extremes, and their changes over time.**

This provides the foundation for analysing climate change in the following lectures.